# Creating Gold Delta Tables from Silver

In [0]:
# Setup database
spark.sql('CREATE DATABASE IF NOT EXISTS gold_wwi')
spark.sql('USE gold_wwi')
spark.sql('DROP TABLE IF EXISTS stockitems')

# Writing the table
dfStockItems = spark.sql("select * from workspace.silver_wwi.stockitems")
dfStockItems.write.format("delta").partitionBy("update_date").option("overwriteSchema", "true").mode("append").saveAsTable("fact_stockitems")


In [0]:
dfStockItems = spark.sql("select * from workspace.gold_wwi.fact_stockitems")
dfStockItems.display()

## Optmizing a table

Creates a big file from smaller files

`deltaTable.optimize().executeCompaction()` is a file compaction operation in Delta Lake.

**Purpose**: It combines many small files in the Delta table into fewer, larger files — a process often called "bin-picking" or "file compaction."

**Why it matters:** Over time, Delta tables accumulate many small files due to frequent appends, updates, and deletes. This "small file problem" degrades read performance because Spark must open and scan many tiny files, adding overhead. Compaction merges those small files into larger ones (typically targeting ~1 GB each), which means:

- Fewer files to scan — Spark reads fewer, larger files instead of many small ones
- Reduced metadata overhead — the Delta log tracks fewer file entries
- Improved query performance — especially for full-scan and range-filtered queries


In [0]:
from delta.tables import *
deltaTable = DeltaTable.forName(spark, "workspace.gold_wwi.fact_stockitems")
deltaTable.optimize().executeCompaction()

## Z-order by columns

Grouping data together by frequently used data sets

`deltaTable.optimize().executeZOrderBy("column_name")` performs Z-ordering (also called multidimensional clustering) on the Delta table.

**Purpose:** It reorganizes the data within the table's files so that rows with similar values for the specified column(s) are stored physically close together.

**How it works:**

- Z-ordering uses a space-filling curve (Z-curve) to interleave the bit representations of one or more column values
- This places rows that share similar values for the Z-ordered column(s) into the same files — effectively co-locating related data
- It also compacts small files in the same pass (like executeCompaction())

**Why it matters — data skipping:**

- Delta Lake automatically collects min/max statistics for each file (for the first 32 columns)
- When you query with a filter like WHERE SupplierID = 123, Spark can skip entire files whose min/max range doesn't include 123
- Without Z-ordering, rows for SupplierID = 123 might be scattered across many files — so Spark can't skip much and must read them all
- With Z-ordering, those rows are clustered into a few files — so Spark can skip the vast majority of files and read only the relevant ones

**Key details:**

- You can Z-order by multiple columns: executeZOrderBy("col1", "col2") — but effectiveness diminishes beyond 2–3 columns
- Best for columns with high cardinality that are frequently used in filter/join predicates
- It's a rewrite operation — it creates new, better-organized files (the old ones become unreferenced and can be cleaned up with vacuum(), as in Cell 9)
- On Databricks, liquid clustering is the newer, recommended alternative to Z-ordering — it handles incremental writes and multi-column clustering more efficiently, but Z-ordering remains widely used and fully supported


In [0]:
from delta.tables import *
deltaTable = DeltaTable.forName(spark, "workspace.gold_wwi.fact_stockitems")
deltaTable.optimize().executeZOrderBy("SupplierID")

## Vacumm tables

Cleanup old snapshot caused by snapshot isolation

`deltaTable.vacuum()` removes old, unreferenced files from the Delta table's storage directory.

**Purpose:** Delta Lake uses snapshot isolation — every time you write, update, or optimize the table, it creates a new version (snapshot) and keeps the old data files on disk so that concurrent readers can still access prior versions. Over time, these orphaned files accumulate and waste storage. vacuum() cleans them up.

**What it actually removes:**

- Files that are no longer referenced by the current Delta transaction log — i.e., files from old snapshots that have been superseded by newer writes, compactions, or Z-ordering operations
- It does not delete files that are still actively referenced by any active snapshot

**Key details:**

- **Default retention threshold:** By default, vacuum() only deletes files older than 7 days (the delta.deletedFileRetentionDuration.checkpoint safety property). This prevents deleting files that might still be needed by active readers or time-travel queries.
- **Dry run:** You can preview what would be deleted without actually removing anything: deltaTable.vacuum(0) with retentionCheckEnabled = False (not recommended for production — it bypasses the safety check and can delete files still needed for time travel).
- **Time travel impact:** After vacuuming, you cannot time-travel to versions whose files have been removed — the history of those old snapshots is permanently gone.
- **It's a storage optimization, not a query optimization** — unlike compaction and Z-ordering which improve read performance, vacuum purely reduces storage costs.



In [0]:
from delta.tables import *
deltaTable = DeltaTable.forName(spark, "workspace.gold_wwi.fact_stockitems")
deltaTable.vacuum()